In [22]:
# creating database
import sqlite3

conn = sqlite3.connect("backup.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS backup(
id INTEGER PRIMARY KEY AUTOINCREMENT,
backup_date TEXT,
source TEXT,
destination TEXT,
status TEXT
)
""")

conn.commit()

print("Database Created Successfully")


Database Created Successfully


In [23]:
# creation function along with simple gui
from tkinter import *
from tkinter import filedialog, messagebox
import os
import shutil
from datetime import datetime

# function of database creation

def create_backup():

    source = "Source_File"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    os.makedirs("backups", exist_ok=True)

    destination = f"backups/backup_{timestamp}"

    shutil.copytree(
        source,
        destination
    )

    print("Backup Created Successfully")

    monitor_label.config(
        text=f"Monitoring: Backup Success at {timestamp}"
    )

# gui

root = Tk()

monitor_label = Label(
    root,
    text="Monitoring: Waiting..."
)

monitor_label.pack()

btn = Button(
    root,
    text="Create Backup",
    command=create_backup
)

btn.pack()

root.mainloop()

Backup Created Successfully
Backup Created Successfully


In [24]:
# database entry save
import os
import shutil
from datetime import datetime

def create_backup():

    source = "Source_File"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    os.makedirs("backups", exist_ok=True)

    destination = f"backups/backup_{timestamp}"

    shutil.copytree(source, destination)

    cursor.execute("""
    INSERT INTO backup
    (backup_date,source,destination,status)
    VALUES(?,?,?,?)
    """,
    (
        timestamp,
        source,
        destination,
        "Success"
    ))

    conn.commit()

    print("Backup Created Successfully")
create_backup()


Backup Created Successfully


In [25]:
# check the database record
cursor.execute("SELECT * FROM backup")

rows = cursor.fetchall()

for row in rows:
    print(row)



(1, '20260627_133857', 'Source_File', 'backups/backup_20260627_133857', 'Success')
(2, '20260627_133916', 'Source_File', 'backups/backup_20260627_133916', 'Success')
(3, '20260627_133932', 'Source_File', 'backups/backup_20260627_133932', 'Success')
(4, '20260627_133951', 'Source_File', 'backups/backup_20260627_133951', 'Success')
(5, '20260627_135907', 'Source_File', 'backups/backup_20260627_135907', 'Success')
(6, '20260627_135930', 'Source_File', 'backups/backup_20260627_135930', 'Success')
(7, '20260627_135946', 'Source_File', 'backups/backup_20260627_135946', 'Success')
(8, '20260627_140219', 'Source_File', 'backups/backup_20260627_140219', 'Success')


In [30]:
import os
import shutil
import logging
from datetime import datetime
from tkinter import *
from tkinter import filedialog, messagebox


# logging
logging.basicConfig(
    filename="backup.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

#function of create database along with mointer label and message box

def create_backup():

    try:

        source = "Source_File"

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        os.makedirs("backups", exist_ok=True)

        destination = f"backups/backup_{timestamp}"

        shutil.copytree(source, destination)

        cursor.execute("""
        INSERT INTO backup
        (backup_date,source,destination,status)
        VALUES(?,?,?,?)
        """,
        (
            timestamp,
            source,
            destination,
            "Success"
        ))

        conn.commit()

        logging.info(
            f"Backup Created: {destination}"
        )

        status_label.config(
            text="Status: Backup Created Successfully"
        )

        monitor_label.config(
            text=f"Monitoring: Backup Success at {timestamp}"
        )

        messagebox.showinfo(
            "Success",
            "Backup Created Successfully"
        )

    except Exception as e:

        logging.error(str(e))

        monitor_label.config(
            text="Monitoring: Backup Failed"
        )

        messagebox.showerror(
            "Error",
            str(e)
        )

# restore function for the restore button
def restore_backup():

    backup_folder = filedialog.askdirectory(
        title="Select Backup Folder"
    )

    if backup_folder:

        destination = "restored_files"

        if os.path.exists(destination):
            shutil.rmtree(destination)

        shutil.copytree(
            backup_folder,
            destination
        )

        logging.info(
            f"Backup Restored: {backup_folder}"
        )

        status_label.config(
            text="Status: Backup Restored"
        )

        messagebox.showinfo(
            "Success",
            "Backup Restored Successfully"
        )

# view history function for the view history button

def view_history():

    cursor.execute(
        "SELECT * FROM backup"
    )

    rows = cursor.fetchall()

    history_text.delete(
        1.0,
        END
    )

    if len(rows) == 0:

        history_text.insert(
            END,
            "No Backup Records Found\n"
        )

    else:

        for row in rows:

            history_text.insert(
                END,
                f"ID:{row[0]} | Date:{row[1]} | Status:{row[4]}\n"
            )

# view_log function for view log button
def view_logs():

    history_text.delete(
        1.0,
        END
    )

    try:

        with open(
            "backup.log",
            "r"
        ) as file:

            history_text.insert(
                END,
                file.read()
            )

    except:

        history_text.insert(
            END,
            "No Logs Found"
        )

# view_report function for view report function
def view_report():

    cursor.execute(
        "SELECT COUNT(*) FROM backup"
    )

    total_backups = cursor.fetchone()[0]

    history_text.delete(
        1.0,
        END
    )

    history_text.insert(
        END,
        f"Total Backups : {total_backups}\n"
    )


def start_auto_backup():

    create_backup()

    root.after(
        60000,
        start_auto_backup
    )

# gui for this veiw a simple dasbord

root = Tk()

root.title("Automated Backup System")
root.geometry("750x600")
root.configure(bg="#F3E8FF")

title = Label(
    root,
    text="Automated Backup System",
    font=("Arial",20,"bold"),
    bg="#EAF6FF",
    fg="navy"
)
title.pack(pady=15)

status_label = Label(
    root,
    text="Status: Ready",
    font=("Arial",12,"bold"),
    bg="#EAF6FF"
)
status_label.pack()

monitor_label = Label(
    root,
    text="Monitoring: Waiting...",
    font=("Arial",11,"bold"),
    bg="#EAF6FF",
    fg="purple"
)
monitor_label.pack(pady=5)

Button(
    root,
    text="Create Backup",
    command=create_backup,
    bg="green",
    fg="white",
    width=20,
    height=2
).pack(pady=5)

Button(
    root,
    text="Restore Backup",
    command=restore_backup,
    bg="orange",
    fg="white",
    width=20,
    height=2
).pack(pady=5)

Button(
    root,
    text="View History",
    command=view_history,
    bg="blue",
    fg="white",
    width=20,
    height=2
).pack(pady=5)

Button(
    root,
    text="View Logs",
    command=view_logs,
    bg="purple",
    fg="white",
    width=20,
    height=2
).pack(pady=5)

Button(
    root,
    text="View Report",
    command=view_report,
    bg="brown",
    fg="white",
    width=20,
    height=2
).pack(pady=5)

Button(
    root,
    text="Start Auto Backup",
    command=start_auto_backup,
    bg="red",
    fg="white",
    width=20,
    height=2
).pack(pady=5)

history_text = Text(
    root,
    width=80,
    height=12
)

history_text.pack(pady=10)

root.mainloop()



In [27]:
# connect backup with database
def create_backup():

    source = "Source_File"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    os.makedirs("backups", exist_ok=True)

    destination = f"backups/backup_{timestamp}"

    shutil.copytree(source, destination)

    cursor.execute("""
    INSERT INTO backup
    (backup_date,source,destination,status)
    VALUES(?,?,?,?)
    """,
    (
        timestamp,
        source,
        destination,
        "Success"
    ))

    conn.commit()

    status_label.config(
        text="Status: Backup Created"
    )

    messagebox.showinfo(
        "Success",
        "Backup Created Successfully"
    )


In [28]:
 # logging
import logging
logging.basicConfig(
    filename = "logs/backups.log",
    level = logging.INFO
)
# backup creation
logging.info(
    "Backup CREATED SUCCESSFULLY"
)
# restore the backup
logging.info(
    "Backup Restored"
)

In [29]:
import os
import shutil
from datetime import datetime
def create_backup():

    source = "Source_File"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    destination = f"backups/backup_{timestamp}"

    shutil.copytree(source, destination)

    print("Folder Copy Done")

    zip_file = shutil.make_archive(
        destination,
        "zip",
        destination
    )

    print("ZIP Created:", zip_file)
create_backup()

Folder Copy Done
ZIP Created: C:\Users\N TECH\Documents\Automated_Backup_system\backups\backup_20260627_140321.zip
